# YSSY Weather Data Processing Pipeline

Sydney Airport (YSSY) area weather station data cleaning, interpolation, and analysis pipeline.

## Pipeline Overview

```
Raw Data (.txt CSV)
    |
    v
Part 1: Merge station data (merge_stations)
    |
    v
Part 2: Crop to year range + Drop unwanted columns
    |
    v
Part 3: Data quality analysis (missing timestamps, outage distribution, availability)
    |
    v
Part 4: Interpolation (linear single-step -> UV conversion -> spline)
    |
    v
Part 5: Verification & summary
```

**Weather Elements**: air_temp, dew_point, wind_speed, wind_dir, max_gust_speed, msl_pressure, aws_flag  
**Time Step**: 30 minutes  
**Wind Convention**: Meteorological (converted to U/V components before spline interpolation)

## Global Configuration

All parameters are defined in the next cell. Folder paths are left blank - fill them in before running.

In [4]:
import pandas as pd
import numpy as np
import os
import glob
import random
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.interpolate import UnivariateSpline

try:
    matplotlib.use('Agg')
except ImportError:
    pass

# ========== Paths (fill in before running) ==========
MERGE_INPUT_FOLDER = ""             # e.g. "ProcessedData Manual"
MERGE_FILE1_NAME = ""               # e.g. "YSRI.txt"
MERGE_FILE2_NAME = ""               # e.g. "YSRI Old.txt"
MERGE_OUTPUT_FILE_NAME = ""         # e.g. "YSRI_merged.txt"

CROP_INPUT_FOLDER = ""              # e.g. "FinalData"
CROP_OUTPUT_FOLDER = ""             # e.g. "FinalData2000"

DROP_INPUT_FOLDER = ""              # e.g. "FinalData"
DROP_OUTPUT_FOLDER = ""             # e.g. "FinalData"

LINEAR_INTERP_INPUT_FOLDER = ""     # e.g. "ProcessedData"
LINEAR_INTERP_OUTPUT_FOLDER = ""    # e.g. "InterpolatedData0.5"

UV_INPUT_FOLDER = ""                # e.g. "InterpolatedData0.5"
UV_OUTPUT_FOLDER = ""               # e.g. "UVComponentData"

SPLINE_INPUT_FOLDER = ""            # e.g. "UVComponentData"
SPLINE_OUTPUT_FOLDER = ""           # e.g. "SplineInterpolatedData"
SPLINE_PLOTS_FOLDER = ""            # e.g. "SplineInterpolationSamplePlots"

OUTAGE_INPUT_FOLDER = ""            # e.g. "InterpolatedData0.5"
OUTAGE_PLOTS_FOLDER = ""            # e.g. "OutageDistributionPlots_Grouped"

AVAILABILITY_INPUT_FOLDER = ""      # e.g. "ProcessedData"
AVAILABILITY_PLOTS_FOLDER = ""      # e.g. "StationElementPlots"

# ========== Run Switches ==========
RUN_MERGE = False
RUN_CROP = False
RUN_DROP = False
RUN_LINEAR_INTERP = False
RUN_UV_CONVERT = False
RUN_SPLINE_INTERP = False

# ========== Data Parameters ==========
START_YEAR_FILTER = None           # e.g. 2000 or None (for UV and spline steps)
CROP_START_YEAR = 2000
CROP_END_YEAR = 2000
TIMESTEP_MINUTES = 30
DATA_START_DATE = pd.Timestamp("2000-01-01 00:00:00")
DATA_END_DATE = pd.Timestamp("2024-12-31 23:30:00")

# ========== Column Configuration ==========
COLUMNS_TO_DROP = ["max_gust_speed", "aws_flag", "wind_dir_recalc"]

WEATHER_PARAMS = [
    "air_temp", "dew_point", "wind_speed", "wind_dir",
    "max_gust_speed", "msl_pressure", "aws_flag"
]

ELEMENTS_TO_INTERPOLATE = [
    "air_temp", "dew_point", "wind_speed",
    "wind_dir", "max_gust_speed", "msl_pressure"
]

ELEMENTS_TO_SPLINE_INTERPOLATE = [
    "air_temp", "dew_point", "msl_pressure",
    "u_component", "v_component"
]

# ========== Spline Parameters ==========
MAX_GAP_STEPS_FOR_SPLINE = 5
SPLINE_CONTEXT_POINTS = 6
SPLINE_ORDER = 3
SPLINE_SMOOTHING_FACTOR = 1
NUM_SAMPLE_PLOTS = 50
PLOT_CONTEXT_HOURS = 12

# ========== Outage Analysis ==========
WEATHER_ELEMENTS_TO_ANALYZE = [
    "air_temp", "dew_point", "wind_speed", "wind_dir", "msl_pressure"
]
OUTAGE_START_YEAR = 2005

print("Global configuration loaded.")

Matplotlib is building the font cache; this may take a moment.


Global configuration loaded.


---
# Part 1: Data Loading & Merging

Merges two station data files (e.g. older/newer station at the same location).  
**Merge rule**: If the newer station (file2) has data, use it; otherwise fall back to older station (file1).

In [ ]:
def read_processed_file(filepath):
    try:
        df = pd.read_csv(
            filepath,
            na_values=['NaN'],
            parse_dates=['timestamp'],
            infer_datetime_format=True
        )
        for col in WEATHER_PARAMS:
            if col in df.columns and df[col].dtype == 'object':
                df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"Successfully read: {filepath}, shape: {df.shape}")
        return df
    except FileNotFoundError:
        print(f"Error: File not found - {filepath}")
        return None
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
        return None


def merge_weather_data(file1_path, file2_path, output_filepath):
    print(f"Starting merge:\n  File 1: {file1_path}\n  File 2: {file2_path}")

    df1 = read_processed_file(file1_path)
    df2 = read_processed_file(file2_path)

    if df1 is None or df2 is None:
        print("One or both input files could not be read. Aborting merge.")
        return None

    merged_df = pd.merge(df1, df2, on="timestamp", how="outer", suffixes=("_f1", "_f2"))
    print(f"Shape after outer merge: {merged_df.shape}")

    for param in WEATHER_PARAMS:
        param_f1 = f"{param}_f1"
        param_f2 = f"{param}_f2"

        if param_f2 in merged_df.columns and param_f1 in merged_df.columns:
            merged_df[param] = merged_df[param_f2].combine_first(merged_df[param_f1])
        elif param_f2 in merged_df.columns:
            merged_df[param] = merged_df[param_f2]
        elif param_f1 in merged_df.columns:
            merged_df[param] = merged_df[param_f1]
        else:
            merged_df[param] = pd.NA

    params_for_completeness = [p for p in WEATHER_PARAMS if p in merged_df.columns]
    merged_df['data_completeness'] = merged_df[params_for_completeness].notna().all(axis=1).astype(int)

    final_columns = ['timestamp'] + WEATHER_PARAMS + ['data_completeness']
    final_columns_present = [col for col in final_columns if col in merged_df.columns]
    output_df = merged_df[final_columns_present].sort_values(by="timestamp").reset_index(drop=True)

    output_dir = os.path.dirname(output_filepath)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    output_df.to_csv(output_filepath, index=False, na_rep='NaN')
    print(f"Merged data saved to: {output_filepath}")
    print(f"Final shape: {output_df.shape}")
    return output_df


print("Merge functions defined.")

In [ ]:
if RUN_MERGE:
    assert MERGE_INPUT_FOLDER, "MERGE_INPUT_FOLDER is empty"
    assert MERGE_FILE1_NAME, "MERGE_FILE1_NAME is empty"
    assert MERGE_FILE2_NAME, "MERGE_FILE2_NAME is empty"
    assert MERGE_OUTPUT_FILE_NAME, "MERGE_OUTPUT_FILE_NAME is empty"

    path_f1 = os.path.join(MERGE_INPUT_FOLDER, MERGE_FILE1_NAME)
    path_f2 = os.path.join(MERGE_INPUT_FOLDER, MERGE_FILE2_NAME)
    path_out = os.path.join(MERGE_INPUT_FOLDER, MERGE_OUTPUT_FILE_NAME)

    if os.path.exists(path_f1) and os.path.exists(path_f2):
        merged = merge_weather_data(path_f1, path_f2, path_out)
        if merged is not None:
            print(f"\nFirst 5 rows:")
            display(merged.head())
    else:
        print(f"File not found.\n  Expected: {path_f1}\n  Expected: {path_f2}")
else:
    print("RUN_MERGE is False. Skipping.")

---
# Part 2: Data Cropping & Column Selection

## Step 2a: Crop to Year Range

Filters all `.txt` files to only include rows within `[CROP_START_YEAR, CROP_END_YEAR]` (inclusive).

In [ ]:
def filter_files_by_year_range(input_folder_path, output_folder_path, start_year, end_year):
    if not os.path.isdir(input_folder_path):
        print(f"Error: Input folder '{input_folder_path}' not found.")
        return

    os.makedirs(output_folder_path, exist_ok=True)
    start_date_dt = pd.Timestamp(f"{start_year}-01-01 00:00:00")
    end_date_dt = pd.Timestamp(f"{end_year}-12-31 23:59:59")

    if start_date_dt > end_date_dt:
        print(f"Error: START_YEAR ({start_year}) cannot be after END_YEAR ({end_year}).")
        return

    print(f"Filtering from {start_date_dt.strftime('%Y-%m-%d')} to {end_date_dt.strftime('%Y-%m-%d')} (inclusive).")

    file_paths = glob.glob(os.path.join(input_folder_path, "*.txt"))
    if not file_paths:
        print(f"No .txt files found in '{input_folder_path}'.")
        return

    print(f"Found {len(file_paths)} .txt files to process.")

    for file_path in file_paths:
        try:
            file_name = os.path.basename(file_path)
            output_file_path = os.path.join(output_folder_path, file_name)
            print(f"\nProcessing '{file_name}'...")

            df = pd.read_csv(file_path)
            if df.empty:
                print("  File is empty. Saving an empty file.")
                df.to_csv(output_file_path, index=False)
                continue

            timestamp_col_name = 'timestamp'
            if timestamp_col_name not in df.columns:
                if df.columns[0]:
                    timestamp_col_name = df.columns[0]
                    print(f"  Warning: 'timestamp' not found. Using first column '{timestamp_col_name}'.")
                else:
                    print(f"  Error: No columns in '{file_name}'. Skipping.")
                    continue

            df[timestamp_col_name] = pd.to_datetime(df[timestamp_col_name], errors='coerce')
            original_rows = len(df)
            df.dropna(subset=[timestamp_col_name], inplace=True)
            if len(df) < original_rows:
                print(f"  Dropped {original_rows - len(df)} rows due to unparseable timestamps.")

            if df.empty:
                df.to_csv(output_file_path, index=False)
                continue

            df_filtered = df[
                (df[timestamp_col_name] >= start_date_dt) &
                (df[timestamp_col_name] <= end_date_dt)
            ]

            if df_filtered.empty:
                print(f"  No data in range ({start_year}-{end_year}).")
            else:
                print(f"  Filtered {len(df_filtered)} rows from {original_rows}.")

            df_filtered.to_csv(output_file_path, index=False)
            print(f"  Saved to '{output_file_path}'")

        except pd.errors.EmptyDataError:
            print(f"  Warning: '{file_name}' is empty. Skipping.")
        except Exception as e:
            print(f"  Error processing '{file_name}': {e}")

    print("\nYear filtering complete.")


print("Crop function defined.")

In [ ]:
if RUN_CROP:
    assert CROP_INPUT_FOLDER, "CROP_INPUT_FOLDER is empty"
    assert CROP_OUTPUT_FOLDER, "CROP_OUTPUT_FOLDER is empty"
    filter_files_by_year_range(CROP_INPUT_FOLDER, CROP_OUTPUT_FOLDER, CROP_START_YEAR, CROP_END_YEAR)
else:
    print("RUN_CROP is False. Skipping.")

## Step 2b: Drop Unwanted Columns

Removes specified columns from all `.txt` files in the input folder.

In [ ]:
def drop_columns_from_files(input_folder_path, output_folder_path, columns_to_drop):
    if not os.path.isdir(input_folder_path):
        print(f"Error: Input folder '{input_folder_path}' not found.")
        return

    os.makedirs(output_folder_path, exist_ok=True)

    file_paths = glob.glob(os.path.join(input_folder_path, "*.txt"))
    if not file_paths:
        print(f"No .txt files found in '{input_folder_path}'.")
        return

    print(f"Found {len(file_paths)} .txt files. Dropping columns: {columns_to_drop}")

    for file_path in file_paths:
        try:
            file_name = os.path.basename(file_path)
            output_file_path = os.path.join(output_folder_path, file_name)
            print(f"\nProcessing '{file_name}'...")

            df = pd.read_csv(file_path)
            existing_cols = [col for col in columns_to_drop if col in df.columns]

            if not existing_cols:
                print(f"  None of {columns_to_drop} found. Saving as is.")
            else:
                print(f"  Dropping: {existing_cols}")

            df_modified = df.drop(columns=existing_cols, errors='ignore')
            df_modified.to_csv(output_file_path, index=False)
            print(f"  Saved to '{output_file_path}'")

        except pd.errors.EmptyDataError:
            print(f"  Warning: '{file_name}' is empty. Skipping.")
        except Exception as e:
            print(f"  Error processing '{file_name}': {e}")

    print("\nColumn dropping complete.")


print("Drop columns function defined.")

In [ ]:
if RUN_DROP:
    assert DROP_INPUT_FOLDER, "DROP_INPUT_FOLDER is empty"
    assert DROP_OUTPUT_FOLDER, "DROP_OUTPUT_FOLDER is empty"
    drop_columns_from_files(DROP_INPUT_FOLDER, DROP_OUTPUT_FOLDER, COLUMNS_TO_DROP)
else:
    print("RUN_DROP is False. Skipping.")

---
# Part 3: Data Quality Analysis

These cells diagnose data completeness before interpolation. They can be run independently.

## Step 3a: Find Missing Timestamps

Compares station files against an expected 30-minute timestamp range (2000-2024).  
Set `COMPARE_FILE1` and `COMPARE_FILE2` to compare two specific stations.

In [ ]:
def find_missing_timestamps(data_folder, file1_name, file2_name=None,
                            start_date=None, end_date=None, timestep_min=30):
    """
    Analyses missing timestamps in station files compared to an expected regular range
    and optionally compares two station files against each other.
    """
    if start_date is None:
        start_date = DATA_START_DATE
    if end_date is None:
        end_date = DATA_END_DATE

    expected_range = pd.date_range(start=start_date, end=end_date, freq=f"{timestep_min}T")
    expected_timestamps = set(expected_range)
    print(f"Expected timestamps in range: {len(expected_timestamps)}")

    df1_path = os.path.join(data_folder, file1_name)
    try:
        df1 = pd.read_csv(df1_path)
        df1['timestamp'] = pd.to_datetime(df1['timestamp'])
        df1 = df1.set_index('timestamp')
        print(f"Loaded {file1_name}, shape: {df1.shape}")
    except Exception as e:
        print(f"Error loading {file1_name}: {e}")
        return

    df1_timestamps = set(df1.index)
    print(f"Unique timestamps in {file1_name}: {len(df1_timestamps)}")

    missing_in_df1 = sorted(list(expected_timestamps - df1_timestamps))
    print(f"\nTimestamps missing in {file1_name} (vs expected range): {len(missing_in_df1)}")
    if missing_in_df1:
        print("First 20 missing:")
        for ts in missing_in_df1[:20]:
            print(f"  {ts}")
        if len(missing_in_df1) > 20:
            print(f"  ... ({len(missing_in_df1) - 20} more)")

    extra_in_df1 = sorted(list(df1_timestamps - expected_timestamps))
    if extra_in_df1:
        print(f"\nWARNING: Timestamps in {file1_name} but NOT in expected range: {len(extra_in_df1)}")

    if file2_name:
        df2_path = os.path.join(data_folder, file2_name)
        try:
            df2 = pd.read_csv(df2_path)
            df2['timestamp'] = pd.to_datetime(df2['timestamp'])
            df2 = df2.set_index('timestamp')
            print(f"\nLoaded {file2_name}, shape: {df2.shape}")
        except Exception as e:
            print(f"Error loading {file2_name}: {e}")
            return

        df2_timestamps = set(df2.index)
        print(f"Unique timestamps in {file2_name}: {len(df2_timestamps)}")

        missing_f1_vs_f2 = sorted(list(df2_timestamps - df1_timestamps))
        print(f"\nTimestamps in {file2_name} but missing in {file1_name}: {len(missing_f1_vs_f2)}")

        missing_f2_vs_f1 = sorted(list(df1_timestamps - df2_timestamps))
        print(f"Timestamps in {file1_name} but missing in {file2_name}: {len(missing_f2_vs_f1)}")


print("Missing timestamps function defined.")

In [ ]:
# Example usage - modify file names as needed
# find_missing_timestamps(
#     data_folder="",
#     file1_name="YSNW.txt",
#     file2_name="YSCN.txt"
)
print("Uncomment and configure the above call to run missing timestamp analysis.")

## Step 3b: Outage Length Distribution

Analyses the distribution of consecutive NaN gaps (outages) across weather elements.  
Generates grouped bar charts showing outage counts by duration category, plus uptime statistics.

In [ ]:
OUTAGE_BINS_CONFIG = [
    (1, 2, "30 min"),
    (2, 3, "1 hr"),
    (3, 5, "1.5-2 hr"),
    (5, 13, "2-6 hr"),
    (13, 49, "6-24 hr"),
    (49, 337, "1-7 day"),
    (337, 1441, "7day-1mo"),
    (1441, 17521, "1mo-1yr"),
    (17521, float('inf'), ">1 yr")
]
OUTAGE_BIN_LABELS = [b[2] for b in OUTAGE_BINS_CONFIG]


def find_outage_durations(series):
    outages = []
    is_na = series.isna()
    current_outage_length = 0
    for val_is_na in is_na:
        if val_is_na:
            current_outage_length += 1
        else:
            if current_outage_length > 0:
                outages.append(current_outage_length)
            current_outage_length = 0
    if current_outage_length > 0:
        outages.append(current_outage_length)
    return outages


def categorize_outages(outage_durations_steps, bins_def):
    bin_counts = {b[2]: 0 for b in bins_def}
    for duration in outage_durations_steps:
        for lower, upper, label in bins_def:
            if lower <= duration < upper:
                bin_counts[label] += 1
                break
    return pd.Series(bin_counts, index=[b[2] for b in bins_def])


def plot_station_grouped_outage_summary(folder_path, output_folder, start_year=None):
    file_paths = glob.glob(os.path.join(folder_path, "*.txt"))
    if not file_paths:
        print(f"No .txt files found in '{folder_path}'.")
        return

    os.makedirs(output_folder, exist_ok=True)

    elements = WEATHER_ELEMENTS_TO_ANALYZE
    num_elements = len(elements)
    num_bins = len(OUTAGE_BIN_LABELS)
    bar_width = 0.8 / num_elements
    element_colors = plt.cm.get_cmap('viridis', num_elements)

    for filepath in file_paths:
        station_id = os.path.basename(filepath).split('.')[0]
        print(f"\n--- Analyzing Station: {station_id} ---")

        try:
            df = pd.read_csv(filepath, parse_dates=['timestamp'], na_values=['NaN'])
            if df.empty:
                continue

            if start_year:
                df_filtered = df[df['timestamp'].dt.year >= start_year].copy()
                if df_filtered.empty:
                    continue
            else:
                df_filtered = df.copy()

            total_obs = len(df_filtered)
            if total_obs == 0:
                continue

            all_binned = pd.DataFrame(index=OUTAGE_BIN_LABELS)
            uptime_stats = {}

            for element in elements:
                if element not in df_filtered.columns:
                    all_binned[element] = 0
                    uptime_stats[element] = "N/A (No Data)"
                    continue

                non_missing = df_filtered[element].notna().sum()
                uptime_pct = (non_missing / total_obs) * 100
                uptime_stats[element] = f"{uptime_pct:.1f}%"

                outage_lengths = find_outage_durations(df_filtered[element])
                if outage_lengths:
                    all_binned[element] = categorize_outages(outage_lengths, OUTAGE_BINS_CONFIG)
                else:
                    all_binned[element] = 0

            fig, ax = plt.subplots(figsize=(16, 8))
            x = np.arange(num_bins)

            if len(all_binned.index) > 1:
                max_y = all_binned.iloc[1:].max().max()
            else:
                max_y = 10
            dynamic_y_lim = max(10, max_y * 1.25)
            ax.set_ylim(0, dynamic_y_lim)

            for i, element in enumerate(elements):
                if element not in all_binned.columns:
                    continue
                counts = all_binned[element].values
                offset = bar_width * (i - num_elements / 2 + 0.5)
                positions = x + offset

                bars = ax.bar(positions, counts, bar_width,
                              label=element.replace("_", " ").title(),
                              color=element_colors(i / num_elements))

                for bar_idx, bar_obj in enumerate(bars):
                    yval = bar_obj.get_height()
                    if yval > 0:
                        is_clipped = (bar_idx == 0 and yval > dynamic_y_lim)
                        if is_clipped:
                            ax.text(bar_obj.get_x() + bar_obj.get_width() / 2.0,
                                    dynamic_y_lim * 0.95,
                                    f'{int(yval)}\n\u2191', ha='center', va='top',
                                    fontsize=7, color='red',
                                    bbox=dict(facecolor='white', alpha=0.6, pad=0.5, edgecolor='none'))
                        else:
                            ax.text(bar_obj.get_x() + bar_obj.get_width() / 2.0, yval,
                                    int(yval), ha='center', va='bottom', fontsize=7)

            title_str = f'Outage Length Distribution - Station: {station_id}'
            if start_year:
                title_str += f' (Since {start_year})'
            ax.set_title(title_str, fontsize=16, pad=20)
            ax.set_xlabel('Outage Duration Category', fontsize=12)
            ax.set_ylabel('Number of Outages', fontsize=12)
            ax.set_xticks(x)
            ax.set_xticklabels(OUTAGE_BIN_LABELS, rotation=45, ha="right", fontsize=10)
            ax.tick_params(axis='y', labelsize=10)
            ax.grid(axis='y', linestyle='--', alpha=0.6)
            ax.legend(title="Weather Element", bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

            uptime_text = "Average Uptime:\n" + "\n".join(
                [f"{el.replace('_', ' ').title()}: {uptime_stats.get(el, 'N/A')}" for el in elements]
            )
            fig.text(0.01, 0.98, uptime_text, transform=fig.transFigure, fontsize=8,
                     verticalalignment='top', horizontalalignment='left',
                     bbox=dict(boxstyle='round,pad=0.5', fc='lightyellow', alpha=0.7))

            plt.tight_layout(rect=[0.05, 0, 0.85, 0.95])
            plot_filename = os.path.join(output_folder, f"{station_id}_grouped_outages.png")
            plt.savefig(plot_filename)
            print(f"  Plot saved: {plot_filename}")
            plt.show()
            plt.close(fig)

        except Exception as e:
            print(f"  Error processing {station_id}: {e}")
            import traceback
            traceback.print_exc()

    print("\nOutage analysis complete.")


print("Outage analysis functions defined.")

In [ ]:
# Example usage - set folders and run
# if OUTAGE_INPUT_FOLDER and OUTAGE_PLOTS_FOLDER:
#     plot_station_grouped_outage_summary(
#         OUTAGE_INPUT_FOLDER, OUTAGE_PLOTS_FOLDER,
#         start_year=OUTAGE_START_YEAR
#     )
print("Uncomment and configure to run outage analysis.")

## Step 3c: Monthly Data Availability

Calculates and plots monthly availability percentages for each weather element, based on full-hour observations.

In [ ]:
WEATHER_ELEMENTS_TO_PLOT = [
    "air_temp", "dew_point", "wind_speed", "wind_dir", "msl_pressure"
]


def plot_station_element_availability(folder_path, output_folder="StationPlots"):
    file_paths = glob.glob(os.path.join(folder_path, "*.txt"))
    if not file_paths:
        print(f"No .txt files found in '{folder_path}'.")
        return

    os.makedirs(output_folder, exist_ok=True)

    for filepath in file_paths:
        station_id = os.path.basename(filepath).split('.')[0]
        print(f"\n--- Processing Station: {station_id} ---")

        try:
            df = pd.read_csv(filepath, parse_dates=['timestamp'], na_values=['NaN'])

            if 'timestamp' not in df.columns:
                print(f"Skipping {station_id}: 'timestamp' column missing.")
                continue
            for element in WEATHER_ELEMENTS_TO_PLOT:
                if element not in df.columns:
                    print(f"Skipping {station_id}: '{element}' missing.")
                    continue

            if df.empty:
                continue

            df_full_hour = df[df['timestamp'].dt.minute == 0].copy()
            if df_full_hour.empty:
                print(f"Skipping {station_id}: No full-hour data.")
                continue

            print(f"  Original: {len(df)}, Full-hour: {len(df_full_hour)}")
            df_full_hour['year_month'] = df_full_hour['timestamp'].dt.to_period('M')

            plt.figure(figsize=(15, 8))
            plot_has_data = False

            for element in WEATHER_ELEMENTS_TO_PLOT:
                if element not in df_full_hour.columns:
                    continue

                monthly = df_full_hour.groupby('year_month').agg(
                    total=('timestamp', 'count'),
                    available=(element, lambda x: x.notna().sum())
                ).reset_index()

                if monthly.empty:
                    continue

                monthly['pct'] = 0.0
                mask = monthly['total'] > 0
                monthly.loc[mask, 'pct'] = (monthly.loc[mask, 'available'] / monthly.loc[mask, 'total']) * 100
                monthly['plot_date'] = monthly['year_month'].dt.to_timestamp()

                plt.plot(monthly['plot_date'], monthly['pct'],
                         label=element.replace('_', ' ').title(), marker='.', linestyle='-')
                plot_has_data = True

            if not plot_has_data:
                plt.close()
                continue

            plt.title(f'Monthly Data Availability - Station: {station_id} (Full Hour)', fontsize=16)
            plt.xlabel('Month', fontsize=14)
            plt.ylabel('Available Observations (%)', fontsize=14)
            plt.ylim(0, 105)
            plt.grid(True, which='major', linestyle='--', linewidth=0.5)
            plt.legend(loc='best', title='Weather Element')

            ax = plt.gca()
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
            ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=8, maxticks=20))
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()

            output_path = os.path.join(output_folder, f"{station_id}_element_availability.png")
            plt.savefig(output_path)
            print(f"  Plot saved: {output_path}")
            plt.show()
            plt.close()

        except Exception as e:
            print(f"  Error processing {station_id}: {e}")
            import traceback
            traceback.print_exc()

    print("\nAvailability analysis complete.")


print("Availability plotting function defined.")

In [ ]:
# Example usage
# if AVAILABILITY_INPUT_FOLDER and AVAILABILITY_PLOTS_FOLDER:
#     plot_station_element_availability(AVAILABILITY_INPUT_FOLDER, AVAILABILITY_PLOTS_FOLDER)
print("Uncomment and configure to run availability analysis.")

---
# Part 4: Interpolation

Three-stage interpolation pipeline:
1. **Linear**: Single-step gaps only (1 missing observation)
2. **UV Conversion**: Wind speed/direction -> U/V components
3. **Spline**: Larger gaps up to `MAX_GAP_STEPS_FOR_SPLINE` steps

## Step 4a: Linear Interpolation (Single-Step Gaps)

Fills gaps of exactly 1 missing timestep using simple linear interpolation `(prev + next) / 2`.

In [ ]:
def interpolate_single_step_gaps(series):
    interpolated_series = series.copy()
    is_na = series.isna()
    n = len(series)
    gaps_count = 0

    for i in range(1, n - 1):
        if not is_na.iloc[i - 1] and is_na.iloc[i] and not is_na.iloc[i + 1]:
            interpolated_series.iloc[i] = (series.iloc[i - 1] + series.iloc[i + 1]) / 2.0
            gaps_count += 1

    if gaps_count > 0:
        print(f"    Interpolated {gaps_count} single-step gaps for '{series.name}'.")
    return interpolated_series


def process_and_interpolate_files(input_folder, output_folder, start_year=None):
    file_paths = glob.glob(os.path.join(input_folder, "*.txt"))
    if not file_paths:
        print(f"No .txt files found in '{input_folder}'.")
        return

    os.makedirs(output_folder, exist_ok=True)

    for filepath in file_paths:
        filename = os.path.basename(filepath)
        station_id = filename.split('.')[0]
        print(f"\n--- Processing Station: {station_id} ---")

        try:
            df = pd.read_csv(filepath, parse_dates=['timestamp'], na_values=['NaN'])
            if df.empty:
                print(f"  Skipping {station_id}: File is empty.")
                continue

            if start_year:
                original_rows = len(df)
                df = df[df['timestamp'].dt.year >= start_year].copy()
                if df.empty:
                    print(f"  Skipping {station_id}: No data after {start_year}.")
                    continue
                print(f"  Year filter: {len(df)} rows (from {original_rows})")

            for element in ELEMENTS_TO_INTERPOLATE:
                if element in df.columns:
                    print(f"  Interpolating '{element}'...")
                    if not pd.api.types.is_numeric_dtype(df[element]):
                        df[element] = pd.to_numeric(df[element], errors='coerce')
                    df[element] = interpolate_single_step_gaps(df[element])
                else:
                    print(f"  Warning: '{element}' not found. Skipping.")

            if 'data_completeness' in df.columns:
                print("  Recalculating 'data_completeness'...")
                cols_check = [el for el in ELEMENTS_TO_INTERPOLATE if el in df.columns]
                if cols_check:
                    df['data_completeness'] = (
                        df[cols_check].notna().all(axis=1) & df['timestamp'].notna()
                    ).astype(int)

            output_filepath = os.path.join(output_folder, filename)
            df.to_csv(output_filepath, index=False, na_rep='NaN', float_format='%.1f')
            print(f"  Saved: {output_filepath}")

        except Exception as e:
            print(f"  Error processing {filepath}: {e}")
            import traceback
            traceback.print_exc()

    print("\nLinear interpolation complete.")


print("Linear interpolation functions defined.")

In [ ]:
if RUN_LINEAR_INTERP:
    assert LINEAR_INTERP_INPUT_FOLDER, "LINEAR_INTERP_INPUT_FOLDER is empty"
    assert LINEAR_INTERP_OUTPUT_FOLDER, "LINEAR_INTERP_OUTPUT_FOLDER is empty"
    process_and_interpolate_files(
        LINEAR_INTERP_INPUT_FOLDER,
        LINEAR_INTERP_OUTPUT_FOLDER,
        start_year=START_YEAR_FILTER
    )
else:
    print("RUN_LINEAR_INTERP is False. Skipping.")

## Step 4b: Wind Speed/Direction -> U/V Components

Converts wind speed and direction to U (eastward) and V (northward) vector components using meteorological convention:
- `u = -speed * sin(direction)`
- `v = speed * cos(direction)`

Original wind_speed and wind_dir columns are replaced.

In [ ]:
def convert_to_uv_and_save(input_folder, output_folder, start_year=None):
    file_paths = glob.glob(os.path.join(input_folder, "*.txt"))
    if not file_paths:
        print(f"No .txt files found in '{input_folder}'.")
        return

    os.makedirs(output_folder, exist_ok=True)

    wind_speed_col = "wind_speed"
    wind_dir_col = "wind_dir"

    for filepath in file_paths:
        filename = os.path.basename(filepath)
        station_id = filename.split('.')[0]
        print(f"\n--- Processing Station: {station_id} ---")

        try:
            df = pd.read_csv(filepath, parse_dates=['timestamp'], na_values=['NaN'])
            if df.empty:
                print(f"  Skipping {station_id}: Empty.")
                continue

            if start_year:
                original_rows = len(df)
                df = df[df['timestamp'].dt.year >= start_year].copy()
                if df.empty:
                    continue
                print(f"  Year filter: {len(df)} rows (from {original_rows})")

            if wind_speed_col not in df.columns or wind_dir_col not in df.columns:
                print(f"  Warning: Wind columns missing in {filename}. Saving as is.")
                df.to_csv(os.path.join(output_folder, filename), index=False, na_rep='NaN', float_format='%.2f')
                continue

            df[wind_speed_col] = pd.to_numeric(df[wind_speed_col], errors='coerce')
            df[wind_dir_col] = pd.to_numeric(df[wind_dir_col], errors='coerce')

            dir_rad = np.deg2rad(df[wind_dir_col].astype(float))

            df['u_component'] = -df[wind_speed_col] * np.sin(dir_rad)
            df['v_component'] = df[wind_speed_col] * np.cos(dir_rad)

            mask_nan_wind = df[wind_speed_col].isna() | df[wind_dir_col].isna()
            df.loc[mask_nan_wind, 'u_component'] = np.nan
            df.loc[mask_nan_wind, 'v_component'] = np.nan

            print(f"  U/V components calculated. Sample U: {df['u_component'].head(3).values}")

            columns_to_keep = [col for col in df.columns if col not in [wind_speed_col, wind_dir_col]]
            df_to_save = df[columns_to_keep]

            output_filepath = os.path.join(output_folder, filename)
            df_to_save.to_csv(output_filepath, index=False, na_rep='NaN', float_format='%.2f')
            print(f"  Saved: {output_filepath}")

        except Exception as e:
            print(f"  Error processing {filepath}: {e}")
            import traceback
            traceback.print_exc()

    print("\nUV conversion complete.")


print("UV conversion function defined.")

In [ ]:
if RUN_UV_CONVERT:
    assert UV_INPUT_FOLDER, "UV_INPUT_FOLDER is empty"
    assert UV_OUTPUT_FOLDER, "UV_OUTPUT_FOLDER is empty"
    convert_to_uv_and_save(UV_INPUT_FOLDER, UV_OUTPUT_FOLDER, start_year=START_YEAR_FILTER)
else:
    print("RUN_UV_CONVERT is False. Skipping.")

## Step 4c: Spline Interpolation

Fills NaN gaps of up to `MAX_GAP_STEPS_FOR_SPLINE` consecutive steps using cubic spline interpolation.  
Interpolates U/V components (not raw wind speed/direction), then recalculates wind speed/direction from the interpolated U/V.  
Also generates sample plots showing interpolation quality.

In [ ]:
def cubic_spline_interpolate_gap(series_with_gap, gap_start_iloc, gap_end_iloc,
                                  context_points, k=3, s=0):
    n = len(series_with_gap)
    ctx_before_start = max(0, gap_start_iloc - context_points)
    ctx_before_end = gap_start_iloc - 1
    ctx_after_start = gap_end_iloc + 1
    ctx_after_end = min(n - 1, gap_end_iloc + context_points)

    x_known_ilocs = []
    y_known_values = []

    if ctx_before_end >= ctx_before_start:
        s_before = series_with_gap.iloc[ctx_before_start:ctx_before_end + 1]
        valid_before = s_before.dropna()
        if not valid_before.empty:
            x_known_ilocs.extend([series_with_gap.index.get_loc(idx) for idx in valid_before.index])
            y_known_values.extend(valid_before.values.tolist())

    if ctx_after_end >= ctx_after_start:
        s_after = series_with_gap.iloc[ctx_after_start:ctx_after_end + 1]
        valid_after = s_after.dropna()
        if not valid_after.empty:
            x_known_ilocs.extend([series_with_gap.index.get_loc(idx) for idx in valid_after.index])
            y_known_values.extend(valid_after.values.tolist())

    if len(x_known_ilocs) < k + 1:
        return None

    x_arr = np.array(x_known_ilocs)
    y_arr = np.array(y_known_values)
    sort_order = np.argsort(x_arr)
    x_sorted = x_arr[sort_order]
    y_sorted = y_arr[sort_order]

    unique_x, unique_idx = np.unique(x_sorted, return_index=True)
    if len(unique_x) < k + 1:
        return None

    y_unique = y_sorted[unique_idx]

    try:
        spline = UnivariateSpline(unique_x, y_unique, k=k, s=s)
        gap_ilocs = np.arange(gap_start_iloc, gap_end_iloc + 1)
        interp_values = spline(gap_ilocs)
        return pd.Series(interp_values, index=series_with_gap.index[gap_ilocs])
    except Exception:
        return None


def apply_spline_interpolation_to_series(series_original, series_name="series"):
    series = series_original.copy()
    is_na_original = series_original.isna()
    n = len(series)
    interpolation_events = []
    current_gap_start = -1

    for i in range(n):
        if is_na_original.iloc[i] and current_gap_start == -1:
            current_gap_start = i
        elif (not is_na_original.iloc[i] or i == n - 1) and current_gap_start != -1:
            gap_end = (i - 1) if not is_na_original.iloc[i] else i
            gap_len = gap_end - current_gap_start + 1

            if 0 < gap_len <= MAX_GAP_STEPS_FOR_SPLINE:
                interp_values = cubic_spline_interpolate_gap(
                    series_original, current_gap_start, gap_end,
                    SPLINE_CONTEXT_POINTS, k=SPLINE_ORDER, s=SPLINE_SMOOTHING_FACTOR
                )
                if interp_values is not None and not interp_values.empty:
                    series.loc[interp_values.index] = interp_values.values
                    interpolation_events.append({
                        'time_start_gap': series_original.index[current_gap_start],
                        'time_end_gap': series_original.index[gap_end],
                        'method': f'spline(k={SPLINE_ORDER},s={SPLINE_SMOOTHING_FACTOR},ctx={SPLINE_CONTEXT_POINTS})',
                        'len_steps': gap_len
                    })
            current_gap_start = -1

    return series, interpolation_events


print("Spline core functions defined.")

In [ ]:
PLOT_COLORS = {
    "air_temp": "red",
    "dew_point": "blue",
    "wind_speed_recalc": "black",
    "wind_dir_recalc": "dimgray"
}
ELEMENTS_TO_DISPLAY_ON_PLOT = [
    "air_temp", "dew_point", "wind_speed_recalc", "wind_dir_recalc"
]
PRIMARY_Y_AXIS_ELEMENTS = ["air_temp", "dew_point", "wind_speed_recalc"]


def plot_interpolation_sample(df_plot_context, station_id, interpolated_element_name,
                               gap_event_info, output_folder):
    fig, ax1 = plt.subplots(figsize=(16, 8))

    gap_start_time = gap_event_info['time_start_gap']
    gap_end_time = gap_event_info['time_end_gap']

    if "u_component" in df_plot_context.columns and "v_component" in df_plot_context.columns:
        if "wind_speed_recalc" not in df_plot_context.columns:
            df_plot_context["wind_speed_recalc"] = np.sqrt(
                df_plot_context["u_component"]**2 + df_plot_context["v_component"]**2
            )
        if "wind_dir_recalc" not in df_plot_context.columns:
            dir_rad = np.arctan2(-df_plot_context["u_component"], -df_plot_context["v_component"])
            df_plot_context["wind_dir_recalc"] = (np.rad2deg(dir_rad) + 360) % 360
            if "wind_speed_recalc" in df_plot_context.columns:
                df_plot_context["wind_dir_recalc"] = df_plot_context["wind_dir_recalc"].where(
                    df_plot_context["wind_speed_recalc"] > 0.01, np.nan
                )

    lines_legend = []
    labels_legend = []
    ax2 = None

    for el_name in ELEMENTS_TO_DISPLAY_ON_PLOT:
        if el_name not in df_plot_context.columns:
            continue

        series_to_plot = df_plot_context[el_name]
        color = PLOT_COLORS.get(el_name, 'gray')
        label_base = el_name.replace('_', ' ').title()

        is_target = (
            el_name == interpolated_element_name or
            (el_name == "wind_speed_recalc" and interpolated_element_name in ["u_component", "v_component"]) or
            (el_name == "wind_dir_recalc" and interpolated_element_name in ["u_component", "v_component"])
        )

        time_anchor_before = gap_start_time
        time_anchor_after = gap_end_time

        try:
            if not df_plot_context.index.is_monotonic_increasing:
                df_plot_context = df_plot_context.sort_index()

            start_iloc = df_plot_context.index.searchsorted(gap_start_time, side='left')
            if start_iloc < len(df_plot_context.index) and df_plot_context.index[start_iloc] >= gap_start_time:
                gap_start_iloc_ctx = start_iloc
            elif gap_start_time in df_plot_context.index:
                gap_start_iloc_ctx = df_plot_context.index.get_loc(gap_start_time)
            else:
                temp_diff = (df_plot_context.index - gap_start_time).to_series().abs()
                gap_start_iloc_ctx = df_plot_context.index.get_loc(temp_diff.idxmin())

            end_iloc = df_plot_context.index.searchsorted(gap_end_time, side='right')
            if end_iloc > 0 and df_plot_context.index[end_iloc - 1] <= gap_end_time:
                gap_end_iloc_ctx = end_iloc - 1
            elif gap_end_time in df_plot_context.index:
                gap_end_iloc_ctx = df_plot_context.index.get_loc(gap_end_time)
            else:
                temp_diff = (df_plot_context.index - gap_end_time).to_series().abs()
                gap_end_iloc_ctx = df_plot_context.index.get_loc(temp_diff.idxmin())

            if not (0 <= gap_start_iloc_ctx < len(df_plot_context.index) and
                    0 <= gap_end_iloc_ctx < len(df_plot_context.index) and
                    gap_start_iloc_ctx <= gap_end_iloc_ctx):
                raise ValueError("Invalid gap ilocs for context.")

            anchor_before_iloc = max(0, gap_start_iloc_ctx - 1)
            anchor_after_iloc = min(len(df_plot_context) - 1, gap_end_iloc_ctx + 1)
            time_anchor_before = df_plot_context.index[anchor_before_iloc]
            time_anchor_after = df_plot_context.index[anchor_after_iloc]

        except (KeyError, ValueError):
            pass

        if el_name == "wind_dir_recalc":
            if ax2 is None:
                ax2 = ax1.twinx()
                ax2.set_ylabel("Wind Direction (deg)", fontsize=10, color=PLOT_COLORS.get(el_name, "dimgray"))
                ax2.tick_params(axis='y', labelcolor=PLOT_COLORS.get(el_name, "dimgray"))
                ax2.set_ylim(0, 360)
                ax2.set_yticks(np.arange(0, 361, 45))
            ax_use = ax2

            if is_target:
                thin_before = series_to_plot.loc[series_to_plot.index < time_anchor_before]
                if not thin_before.empty:
                    ax_use.scatter(thin_before.index, thin_before.values, color=color, marker='.', s=15, alpha=0.6)
                thin_after = series_to_plot.loc[series_to_plot.index > time_anchor_after]
                if not thin_after.empty:
                    ax_use.scatter(thin_after.index, thin_after.values, color=color, marker='.', s=15, alpha=0.6)
                influenced = series_to_plot.loc[time_anchor_before:time_anchor_after]
                if not influenced.empty:
                    handle = ax_use.scatter(influenced.index, influenced.values, color=color,
                                             marker='o', s=35, label=label_base + " (from Interp. U/V)")
                    lines_legend.append(handle)
                    labels_legend.append(label_base + " (from Interp. U/V)")
            else:
                handle = ax_use.scatter(series_to_plot.index, series_to_plot.values,
                                        color=color, marker='.', s=15, alpha=0.6, label=label_base + " (context)")
                lines_legend.append(handle)
                labels_legend.append(label_base + " (context)")
        else:
            ax_use = ax1
            if is_target:
                series_before = series_to_plot.loc[series_to_plot.index <= time_anchor_before]
                if not series_before.empty:
                    ax_use.plot(series_before.index, series_before.values, color=color, linewidth=1.5, linestyle='-')

                segment_thick = series_to_plot.loc[time_anchor_before:time_anchor_after]
                if not segment_thick.empty:
                    line_thick, = ax_use.plot(segment_thick.index, segment_thick.values,
                                              color=color, linewidth=3.0, linestyle='-',
                                              label=label_base + " (Interpolated Section)")
                    lines_legend.append(line_thick)
                    labels_legend.append(label_base + " (Interpolated Section)")

                series_after = series_to_plot.loc[series_to_plot.index >= time_anchor_after]
                if not series_after.empty:
                    ax_use.plot(series_after.index, series_after.values, color=color, linewidth=1.5, linestyle='-')
            else:
                line, = ax_use.plot(series_to_plot.index, series_to_plot.values,
                                    color=color, linewidth=1.0, linestyle='-', alpha=0.6,
                                    label=label_base + " (context)")
                lines_legend.append(line)
                labels_legend.append(label_base + " (context)")

    ax1.set_xlabel("Timestamp", fontsize=10)
    ax1.set_ylabel("Temp (C), Dewpt (C), Speed (kts)", fontsize=10)

    fig.suptitle(
        f"Spline Interpolation: {interpolated_element_name.replace('_', ' ').title()} for {station_id}\n"
        f"Gap: {gap_start_time} to {gap_end_time} (Method: {gap_event_info['method']})",
        fontsize=12, y=0.98
    )
    ax1.grid(True, linestyle='--', alpha=0.7)
    fig.autofmt_xdate()

    if lines_legend:
        fig.legend(lines_legend, labels_legend, loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8)
        plt.subplots_adjust(right=0.80, top=0.90)
    else:
        plt.subplots_adjust(top=0.90)

    plot_filename = os.path.join(
        output_folder,
        f"{station_id}_{interpolated_element_name}_spline_sample_{gap_event_info['sample_num']}.png"
    )
    plt.savefig(plot_filename)
    plt.show()
    plt.close(fig)


print("Plotting function defined.")

In [ ]:
def run_spline_interpolation_pipeline(input_uv_folder, output_spline_folder,
                                       plot_output_folder, start_year=None):
    file_paths = glob.glob(os.path.join(input_uv_folder, "*.txt"))
    if not file_paths:
        print(f"No .txt files found in '{input_uv_folder}'.")
        return

    os.makedirs(output_spline_folder, exist_ok=True)
    os.makedirs(plot_output_folder, exist_ok=True)

    master_events_log = []

    for filepath in file_paths:
        filename = os.path.basename(filepath)
        station_id = filename.split('.')[0]
        print(f"\n--- Spline Interpolation: {station_id} ---")

        try:
            df_original = pd.read_csv(filepath, parse_dates=['timestamp'], na_values=['NaN'])
            if df_original.empty:
                continue

            df = df_original.copy()
            if start_year:
                df = df[df['timestamp'].dt.year >= start_year].copy()
                if df.empty:
                    continue

            df = df.set_index('timestamp')
            df_to_save = df.copy()
            station_events = []

            for element in ELEMENTS_TO_SPLINE_INTERPOLATE:
                if element not in df.columns:
                    print(f"  '{element}' not found in {station_id}. Skipping.")
                    continue

                print(f"  Spline interpolating '{element}'...")
                if not pd.api.types.is_numeric_dtype(df[element]):
                    df[element] = pd.to_numeric(df[element], errors='coerce')

                interp_series, events = apply_spline_interpolation_to_series(df[element], series_name=element)
                df_to_save[element] = interp_series

                for info in events:
                    info['station_id'] = station_id
                    info['element'] = element
                    station_events.append(info)

            if 'u_component' in df_to_save.columns and 'v_component' in df_to_save.columns:
                print(f"  Recalculating wind from interpolated U/V for {station_id}")
                df_to_save['wind_speed_recalc'] = np.sqrt(
                    df_to_save['u_component']**2 + df_to_save['v_component']**2
                )
                dir_rad = np.arctan2(-df_to_save['u_component'], -df_to_save['v_component'])
                df_to_save['wind_dir_recalc'] = (np.rad2deg(dir_rad) + 360) % 360
                df_to_save['wind_dir_recalc'] = df_to_save['wind_dir_recalc'].where(
                    df_to_save['wind_speed_recalc'].notna() & (df_to_save['wind_speed_recalc'] > 0.01),
                    np.nan
                )

            if 'data_completeness' in df_to_save.columns:
                print(f"  Recalculating 'data_completeness' for {station_id}...")
                temp_cols = ["air_temp", "dew_point", "msl_pressure"]
                if 'wind_speed_recalc' in df_to_save.columns:
                    temp_cols.append('wind_speed_recalc')
                if 'wind_dir_recalc' in df_to_save.columns:
                    temp_cols.append('wind_dir_recalc')
                cols_present = [el for el in temp_cols if el in df_to_save.columns]
                if cols_present:
                    df_to_save['data_completeness'] = df_to_save[cols_present].notna().all(axis=1).astype(int)

            output_filepath = os.path.join(output_spline_folder, filename)
            df_to_save.reset_index().to_csv(output_filepath, index=False, na_rep='NaN', float_format='%.3f')
            print(f"  Saved: {output_filepath}")

            master_events_log.extend(station_events)

        except Exception as e:
            print(f"  Error processing {station_id}: {e}")
            import traceback
            traceback.print_exc()

    # Generate sample plots
    print(f"\n--- Generating Sample Spline Plots ({NUM_SAMPLE_PLOTS} max) ---")
    if not master_events_log:
        print("  No interpolation events recorded. Skipping plots.")
        return

    num_to_plot = min(NUM_SAMPLE_PLOTS, len(master_events_log))
    sampled_events = random.sample(master_events_log, num_to_plot)
    context_delta = pd.Timedelta(hours=PLOT_CONTEXT_HOURS)

    for i, event_info in enumerate(sampled_events):
        sid = event_info['station_id']
        element_interp = event_info['element']
        plot_file = os.path.join(output_spline_folder, f"{sid}.txt")

        if not os.path.exists(plot_file):
            continue

        df_plot = pd.read_csv(plot_file, parse_dates=['timestamp'], na_values=['NaN'])
        df_plot = df_plot.set_index('timestamp')

        plot_start = event_info['time_start_gap'] - context_delta
        plot_end = event_info['time_end_gap'] + context_delta

        try:
            min_t, max_t = df_plot.index.min(), df_plot.index.max()
            actual_start = max(min_t, plot_start)
            actual_end = min(max_t, plot_end)
            ctx_slice = df_plot.loc[actual_start:actual_end]
        except Exception as e:
            print(f"    Error slicing for plot ({sid}, {element_interp}): {e}")
            continue

        if ctx_slice.empty:
            continue

        event_info_copy = event_info.copy()
        event_info_copy['sample_num'] = i + 1
        plot_interpolation_sample(ctx_slice, sid, element_interp, event_info_copy, plot_output_folder)

    print("\n--- Spline interpolation and plotting pipeline complete. ---")


print("Spline pipeline function defined.")

In [ ]:
if RUN_SPLINE_INTERP:
    assert SPLINE_INPUT_FOLDER, "SPLINE_INPUT_FOLDER is empty"
    assert SPLINE_OUTPUT_FOLDER, "SPLINE_OUTPUT_FOLDER is empty"
    assert SPLINE_PLOTS_FOLDER, "SPLINE_PLOTS_FOLDER is empty"
    run_spline_interpolation_pipeline(
        SPLINE_INPUT_FOLDER, SPLINE_OUTPUT_FOLDER,
        SPLINE_PLOTS_FOLDER, start_year=START_YEAR_FILTER
    )
else:
    print("RUN_SPLINE_INTERP is False. Skipping.")

---
# Part 5: Interpolation Verification

Validates the spline interpolation approach on synthetic data.

In [ ]:
def test_spline_on_synthetic_data(n_points=48, gap_start=20, n_missing=10,
                                   n_local_points=5, smoothing=0.4, degree=3):
    """
    Tests local spline interpolation on synthetic time series data.
    """
    np.random.seed(42)
    time = np.arange(n_points)
    trend = np.sin(time / ((n_points - 1) / (2 * np.pi))) + time / 20
    noise = np.random.normal(0, 0.3, n_points)
    data_original = trend + noise

    data_with_missing = data_original.copy()
    data_with_missing[gap_start:gap_start + n_missing] = np.nan

    series_missing = pd.Series(data_with_missing, index=time)
    series_original = pd.Series(data_original, index=time)
    series_interpolated = series_missing.copy()

    gap_indices = series_interpolated.index[gap_start:gap_start + n_missing]

    points_before = series_missing.iloc[:gap_start].dropna().tail(n_local_points)
    points_after = series_missing.iloc[gap_start + n_missing:].dropna().head(n_local_points)
    local_data = pd.concat([points_before, points_after])

    if len(local_data) < degree + 1:
        print(f"Not enough points ({len(local_data)}) for degree {degree}. Falling back to linear.")
        if len(local_data) >= 2:
            temp = pd.concat([points_before, pd.Series(index=gap_indices, dtype=float), points_after])
            filled = temp.interpolate(method='linear').loc[gap_indices]
            series_interpolated.loc[gap_indices] = filled
    else:
        spline = UnivariateSpline(local_data.index.values, local_data.values, s=smoothing, k=degree)
        series_interpolated.loc[gap_indices] = spline(gap_indices.values)

    plt.figure(figsize=(15, 10))
    plt.plot(series_original.index, series_original.values, 'ko-', label='Original', alpha=0.3, markersize=3)
    plt.plot(series_missing.index, series_missing.values, 'bo', label='With Missing Gap', markersize=6)

    if len(local_data) >= degree + 1:
        plt.plot(local_data.index, local_data.values, 'ms', markersize=8,
                 markerfacecolor='none', label=f'Local {n_local_points}x2 points')

    plt.plot(series_interpolated.index, series_interpolated.values, 'gD--',
             label=f'Local Spline (s={smoothing}, k={degree})', linewidth=2, markersize=4)

    gap_end_idx = gap_start + n_missing - 1
    plt.axvline(series_missing.index[gap_start], color='gray', linestyle='--', alpha=0.7, label='Gap Start/End')
    plt.axvline(series_missing.index[gap_end_idx], color='gray', linestyle='--', alpha=0.7)

    plt.title(f'Local Spline Interpolation Test (gap={n_missing} steps, ctx={n_local_points} each side)')
    plt.xlabel('Time Step')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    rmse = np.sqrt(np.mean((series_interpolated.loc[gap_indices].values - series_original.loc[gap_indices].values) ** 2))
    print(f"RMSE of interpolation: {rmse:.4f}")
    return rmse


rmse = test_spline_on_synthetic_data(
    n_points=48, gap_start=20, n_missing=10,
    n_local_points=SPLINE_CONTEXT_POINTS,
    smoothing=SPLINE_SMOOTHING_FACTOR,
    degree=SPLINE_ORDER
)

print(f"\nSynthetic test complete. RMSE: {rmse:.4f}")

---
# Summary

This notebook implements the complete YSSY weather data processing pipeline:

| Step | Description | Input | Output |
|------|-------------|-------|--------|
| 1 | Merge stations | Two station `.txt` files | Merged `.txt` |
| 2a | Crop to years | `.txt` files | Year-filtered `.txt` |
| 2b | Drop columns | `.txt` files | Cleaned `.txt` |
| 3a | Missing timestamps | Station files | Console report |
| 3b | Outage distribution | `.txt` files | Grouped bar charts |
| 3c | Monthly availability | `.txt` files | Availability plots |
| 4a | Linear interpolation | Processed `.txt` | Interpolated `.txt` |
| 4b | UV conversion | Interpolated `.txt` | UV component `.txt` |
| 4c | Spline interpolation | UV `.txt` | Final spline-interpolated `.txt` |
| 5 | Verification | Synthetic data | Test plot + RMSE |

### Key Parameters
- **Max spline gap**: 5 steps (2.5 hours at 30-min intervals)
- **Spline context**: 6 points each side
- **Spline order**: 3 (cubic)
- **Spline smoothing**: 1

### Next Steps
- Fill in folder paths in the configuration cell
- Set `RUN_XXX = True` for steps you want to execute
- Run cells sequentially